
# CNN for MNIST Digit Classification

In this notebook, you will:
- Build and train a Convolutional Neural Network (CNN) on the MNIST dataset.
- Experiment with adding convolutional and pooling layers.
- Reflect on how data size and feature complexity affect CNN design and performance.
- Compare CNN performance to traditional ML methods like SVM.
- Understand how CNN filters learn local spatial patterns.

This hands-on activity helps demonstrate **why CNNs are powerful and interpretable models for image data**.


In [ ]:

# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


In [ ]:

# Load and preprocess the MNIST dataset
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train / 255.0
x_test = x_test / 255.0

# Add channel dimension
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)


In [ ]:

# Visualize some training samples
plt.figure(figsize=(10,4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i].squeeze(), cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:

# 🛠 Build a simple CNN model
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()


In [ ]:

# Compile and train the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(x_train, y_train, epochs=5, 
                    validation_data=(x_test, y_test))


In [ ]:

# Evaluate performance on test data
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test Accuracy: {test_acc:.4f}")


In [ ]:

# Plot confusion matrix
y_pred = np.argmax(model.predict(x_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [ ]:

# Visualize what the first layer is learning
layer_outputs = [layer.output for layer in model.layers[:2]]

# Explicitly build the model to define its input shape
model.build(input_shape=(None, 28, 28, 1))

# Create a new model that outputs the activations of the first two layers
activation_model = models.Model(inputs=model.inputs, outputs=layer_outputs)

# Get the activations for the first test sample
activations = activation_model.predict(x_test[:1])

first_layer_activation = activations[0]
plt.figure(figsize=(15,5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(first_layer_activation[0, :, :, i], cmap='viridis')
    plt.title(f'Filter {i}')
    plt.axis('off')
plt.suptitle('First Convolutional Layer Activations')
plt.tight_layout()
plt.show()



##  Reflection Questions

After running and analyzing the CNN:

- Why did you choose this number of layers and filters?
- How do the convolutional filters contribute to the learning process?
- How does the performance of this CNN compare to traditional ML methods (like SVM)?
- What patterns do the filters learn in early vs deeper layers?

Be prepared to **explain your design choices and insights** to the tutor.
